# Unity Catalog Schema Setup

## Clinical Trial Intelligence Platform

This notebook creates the logical Unity Catalog schemas used by the platform's medallion architecture and data-quality framework.

### Schema architecture

**clinical_trial_intelligence**
- `bronze` — raw source-aligned clinical data
- `silver` — cleaned, standardized and validated clinical data
- `quarantine` — records rejected by blocking Silver data-quality controls
- `gold` — business-ready analytical datasets

The setup is designed to be safely rerunnable through `CREATE SCHEMA IF NOT EXISTS`.

## 1. Bronze Schema

The Bronze schema stores source-aligned data ingested from the clinical operational systems.

Its main responsibilities are:

- preserve incoming source representation
- retain source-level lineage
- provide the controlled input to downstream Silver processing

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS clinical_trial_intelligence.bronze
MANAGED LOCATION 's3://clinical-trial-intelligence-platform-sk/UnityManaged/bronze'
COMMENT 'Raw source data ingested from clinical operational systems';

## 2. Silver Schema

The Silver schema stores standardized and validated clinical data.

This layer applies:

- schema and datatype standardization
- reference-data validation
- clinical business rules
- cross-entity consistency checks
- historical preservation using CDC / SCD Type 2 where required

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS clinical_trial_intelligence.silver
MANAGED LOCATION 's3://clinical-trial-intelligence-platform-sk/UnityManaged/silver'
COMMENT 'Cleaned, standardized, deduplicated and validated clinical data';

## 3. Quarantine Schema

The Quarantine schema retains records that fail blocking Silver data-quality controls.

Rejected records are preserved together with their failure reasons rather than being silently discarded.

This supports:

- data-quality investigation
- auditability
- reconciliation
- transparent downstream reporting

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS clinical_trial_intelligence.quarantine
MANAGED LOCATION 's3://clinical-trial-intelligence-platform-sk/UnityManaged/quarantine'
COMMENT 'Records rejected from Silver processing because of data quality or business-rule violations';

## 5. Schema Verification

Verify that all required schemas have been created under the `clinical_trial_intelligence` catalog.

In [0]:
%sql

DESCRIBE SCHEMA EXTENDED clinical_trial_intelligence.bronze;

In [0]:
%sql

DESCRIBE SCHEMA EXTENDED clinical_trial_intelligence.silver;

In [0]:
%sql

DESCRIBE SCHEMA EXTENDED clinical_trial_intelligence.quarantine;

In [0]:
%sql

DESCRIBE SCHEMA EXTENDED clinical_trial_intelligence.gold;